In [1]:
import pandas as pd
import numpy as np

In [4]:
df=pd.read_csv("gurgaon_properties_post_feature_selection_v2.csv")

In [5]:
df.head(1)

,property_type,sector,price_per_sqft,bedRoom,bathroom,balcony,servant room,age_category,furnishing_type,combined_rating,dist_cyber_city_km,luxury_category,floor_category,price
0,0.0,72.0,5000.0,2,2,1.0,0,3.0,2.0,4,7.932372,1.0,2.0,0.45


In [6]:
x=df.drop(columns=['price'])
y=df['price']

In [7]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR

In [8]:
df.head(1)

,property_type,sector,price_per_sqft,bedRoom,bathroom,balcony,servant room,age_category,furnishing_type,combined_rating,dist_cyber_city_km,luxury_category,floor_category,price
0,0.0,72.0,5000.0,2,2,1.0,0,3.0,2.0,4,7.932372,1.0,2.0,0.45


In [18]:
columns_to_encode = ['sector', 'balcony', 'bedRoom','bathroom','age_category', 'servant room','furnishing_type', 'luxury_category', 'floor_category','combined_rating']

In [19]:
y_transformed = np.log1p(y)

In [25]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['property_type', 'price_per_sqft', 'dist_cyber_city_km']),
        ('cat', OneHotEncoder(drop='first',handle_unknown='ignore'), columns_to_encode)
    ], 
    remainder='passthrough'
)

In [26]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    # ('regressor', LinearRegression())
    ('regressor', SVR(kernel='rbf'))

])

In [27]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, x, y_transformed, cv=kfold, scoring='r2')

c:\Users\LENOVO\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\LENOVO\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\LENOVO\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [28]:
scores.mean()

np.float64(0.8985474164222327)

In [30]:
scores.std()

np.float64(0.01089203961045844)

In [31]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(x,y_transformed,test_size=0.2,random_state=42)

In [32]:
pipeline.fit(X_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tran

In [33]:
y_pred=pipeline.predict(X_test)

In [34]:
y_pred=np.expm1(y_pred)

In [35]:
from sklearn.metrics import mean_absolute_error
mean_absolute_error(np.expm1(y_test),y_pred)

0.5089685881499731

In [37]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

import numpy as np

models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(random_state=42),
    "Lasso": Lasso(random_state=42),
    "ElasticNet": ElasticNet(random_state=42),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Extra Trees": ExtraTreesRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "AdaBoost": AdaBoostRegressor(random_state=42),
    "KNN": KNeighborsRegressor(),
    "SVR": SVR(),

    # Boosting Models
    "XGBoost": XGBRegressor(
        random_state=42,
        verbosity=0
    ),

    "LightGBM": LGBMRegressor(
        random_state=42,
        verbose=-1
    ),

    "CatBoost": CatBoostRegressor(
        random_state=42,
        verbose=0
    )
}

kfold = KFold(n_splits=10, shuffle=True, random_state=42)

print(f'{"Model":25} {"R² Score":10} {"MAE":10}')
print("-"*50)

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    y_pred_log = cross_val_predict(
        pipeline,
        x,
        y_transformed,
        cv=kfold,
        n_jobs=-1
    )

    r2 = r2_score(y_transformed, y_pred_log)

    y_true = np.expm1(y_transformed)
    y_pred = np.expm1(y_pred_log)

    mae = mean_absolute_error(y_true, y_pred)

    print(f"{name:25} {r2:.4f}     {mae:.4f}")

Model                     R² Score   MAE       
--------------------------------------------------
Linear Regression         0.8681     0.6189
Ridge                     0.8694     0.6184
Lasso                     -0.0006     1.5832
ElasticNet                -0.0006     1.5832
Decision Tree             0.7906     0.6700
Random Forest             0.8824     0.5310
Extra Trees               0.8914     0.5032
Gradient Boosting         0.8609     0.6153
AdaBoost                  0.7216     0.8760
KNN                       0.8528     0.6147
SVR                       0.8987     0.5276
XGBoost                   0.8887     0.5351
LightGBM                  0.8895     0.5381


CatBoostError: catboost/libs/train_lib/dir_helper.cpp:26: Can't create train tmp dir: tmp